# N-граммная языковая модель

Построим модель, которая продолжает обращение по частотам соседних слов. Сначала проверим её на трёх предложениях из лекции, затем сравним униграммы, биграммы и триграммы на корпусе «Север». В конце прочитаем несколько продолжений и попробуем объяснить, почему они получились именно такими.

In [1]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Подготовка последовательностей

Берём по одному обращению из каждой группы. Токенизатор выделяет слова и числа; пунктуация в этой работе не предсказывается. `BOS` задаёт начало истории, `EOS` обозначает предсказываемое завершение, `UNK` заменяет слова вне словаря.

Функция `events` уже написана. Она выдаёт пары «история, следующий токен» и начинает историю заново для каждого предложения. Посмотрите на пример её работы перед тем, как писать счётчики.

In [2]:
from collections import Counter, defaultdict
import math

BOS, EOS, UNK = "<BOS>", "<EOS>", "<UNK>"
def tokenize(text):
    return re.findall(r"[а-яёa-z0-9]+", text.lower())

df = pd.read_csv(DATA / "requests.csv", keep_default_na=False).drop_duplicates("group_id")
parts = {name: [tokenize(t) for t in df.loc[df["split"].eq(name), "text"]]
         for name in ["train", "validation", "test"]}
assert tokenize("Студент читает книгу") == ["студент", "читает", "книгу"]
print({name: len(values) for name, values in parts.items()})

# Готовый итератор отделяет устройство истории от подсчёта частот.
def events(sentence, n):
    sequence = [BOS] * (n - 1) + list(sentence) + [EOS]
    for t in range(n - 1, len(sequence)):
        history = tuple(sequence[t - n + 1:t]) if n > 1 else ()
        yield history, sequence[t]


{'train': 72, 'validation': 24, 'test': 24}


In [3]:
demo_sentence = tokenize("студент читает книгу")
for history, target in events(demo_sentence, 2):
    print(history, "→", target)

('<BOS>',) → студент
('студент',) → читает
('читает',) → книгу
('книгу',) → <EOS>


У биграммы в истории один токен. Последняя цель — EOS, а первая история — BOS. Для триграммы перед первым словом появились бы два BOS, но число целей осталось бы тем же.

## 1. Общий словарь

Посчитайте частоты слов только в `parts["train"]`. Сохраните множество слов с частотой не меньше двух в `known`. Создайте отсортированный список `vocab`, добавив `EOS` и `UNK` и исключив `BOS`.

Создайте `encoded`: словарь с ключами train, validation, test и списками токенизированных предложений. Во всех частях заменяйте слова вне `known` на `UNK`. Посчитайте долю UNK среди обычных слов каждой части, без EOS. Что эта доля говорит о сохранённых различиях текста?

In [4]:
# Ваш код
train_counts = Counter()
for sentence in parts["train"]:
    for word in sentence:
        train_counts[word] += 1

known = set()
for word, count in train_counts.items():
    if count >= 2:
        known.add(word)

vocab = sorted(known | {EOS, UNK})


def encode(sentence):
    result = []
    for word in sentence:
        if word in known:
            result.append(word)
        else:
            result.append(UNK)
    return result


encoded = {}
for name in ["train", "validation", "test"]:
    encoded[name] = []
    for sentence in parts[name]:
        encoded[name].append(encode(sentence))

unk_rows = []
for name in ["train", "validation", "test"]:
    words = 0
    unk_words = 0
    for sentence in encoded[name]:
        words += len(sentence)
        unk_words += sentence.count(UNK)
    unk_rows.append({"part": name, "words": words, "unk": unk_words, "unk_share": unk_words / words})
unk_table = pd.DataFrame(unk_rows)

print("Разных слов в train:", len(train_counts))
print("Размер словаря:", len(vocab))
display(unk_table)

Разных слов в train: 309
Размер словаря: 47


,part,words,unk,unk_share
0,train,455,264,0.580220
1,validation,165,113,0.684848
2,test,166,118,0.710843


In [5]:
assert BOS not in vocab and EOS in vocab and UNK in vocab
assert all(t in vocab for sentences in encoded.values() for sentence in sentences for t in sentence)
assert len(encoded["train"]) == 72 and len(encoded["validation"]) == len(encoded["test"]) == 24

**Ответ.** В словаре 47 элементов, хотя в train 309 разных слов. UNK заменил 58% слов train, 68% validation и 71% test, поэтому большая часть различий между исходными словами потеряна.

## 2. Счётчики переходов

Напишите `fit_model(sentences, n, vocab)` с помощью `events`. Для каждой истории считайте отдельные продолжения. Верните словарь с полями `n`, `vocab` (tuple), `counts` (история → Counter токенов), `totals` (история → общее число продолжений).

Проверьте три предложения из лекции. После «студент» должны получиться два «читает» и одно «возвращает». Объясните, почему всего целей двенадцать, хотя обычных слов девять.

In [6]:
def fit_model(sentences, n, vocab):
    # Ваш код
    counts = {}
    totals = {}
    for sentence in sentences:
        for history, target in events(sentence, n):
            if history not in counts:
                counts[history] = Counter()
                totals[history] = 0
            counts[history][target] += 1
            totals[history] += 1
    return {"n": n, "vocab": tuple(vocab), "counts": counts, "totals": totals}

In [7]:
toy_sentences = [tokenize(t) for t in ["студент читает книгу", "студент читает журнал", "студент возвращает книгу"]]
toy_vocab = sorted({t for s in toy_sentences for t in s} | {EOS, UNK})
toy_model = fit_model(toy_sentences, 2, toy_vocab)
assert len(toy_vocab) == 7
assert toy_model["counts"][("студент",)]["читает"] == 2
assert toy_model["totals"][("студент",)] == 3
assert sum(toy_model["totals"].values()) == 12  # 9 слов + 3EOS
assert all(BOS not in continuation for continuation in toy_model["counts"].values())

**Ответ.** Целей 12: девять слов и ещё по одному EOS на каждое из трёх предложений. BOS только начинает историю и целью не бывает.

## 3. Вероятность нового продолжения

Реализуйте `probability(model, history, target, alpha=0.1)` по add-alpha. При `alpha <= 0` или цели вне словаря вызывайте `ValueError`. Для невстречавшейся истории счётчики равны нулю.

Готовая `distribution` собирает значения в порядке `model["vocab"]`. Примените её к истории «студент» при alpha=1 и сверьте семь вероятностей с лекцией. Затем возьмите невстречавшуюся историю. Почему у неё получается равномерное распределение?

In [8]:
def probability(model, history, target, alpha=0.1):
    # Ваш код
    if alpha <= 0:
        raise ValueError("alpha должна быть больше нуля")
    if target not in model["vocab"]:
        raise ValueError("цели нет в словаре")

    history = tuple(history)
    count = 0
    total = 0
    if history in model["counts"]:
        count = model["counts"][history][target]
        total = model["totals"][history]

    vocab_size = len(model["vocab"])
    return (count + alpha) / (total + alpha * vocab_size)

def distribution(model, history, alpha=0.1):
    return np.array([probability(model, history, t, alpha) for t in model["vocab"]])


for history in [("студент",), ("невстречавшаяся_история",)]:
    probs = distribution(toy_model, history, 1)
    table = pd.DataFrame({"token": toy_model["vocab"], "probability": probs})
    print("История:", history)
    display(table)

История: ('студент',)


,token,probability
0,<EOS>,0.1
1,<UNK>,0.1
2,возвращает,0.2
3,журнал,0.1
4,книгу,0.1
5,студент,0.1
6,читает,0.3


История: ('невстречавшаяся_история',)


,token,probability
0,<EOS>,0.142857
1,<UNK>,0.142857
2,возвращает,0.142857
3,журнал,0.142857
4,книгу,0.142857
5,студент,0.142857
6,читает,0.142857


In [9]:
assert np.isclose(probability(toy_model, ("студент",), "читает", 1), 0.3)
assert np.isclose(probability(toy_model, ("студент",), "возвращает", 1), 0.2)
for h in [(BOS,), ("студент",), ("невстречавшаяся_история",)]:
    p = distribution(toy_model, h, 1)
    assert (p > 0).all() and np.isclose(p.sum(), 1)
assert np.allclose(distribution(toy_model, ("невстречавшаяся_история",), 1), np.ones(7)/7)

**Ответ.** После «студент» при α=1: P(читает) = (2+1)/(3+7) = 0,3, P(возвращает) = 0,2, остальные пять токенов по 0,1. У новой истории count = 0 и total = 0, поэтому у всех токенов α/(7α) = 1/7.

## 4. Perplexity и выбор порядка

Напишите `evaluate_ppl(model, sentences, alpha=0.1)`. Суммируйте −ln вероятностей целей, включая EOS. Верните словарь `ppl`, `mean_nll`, `targets`. При отсутствии целевых событий вызывайте `ValueError`; одно пустое предложение всё ещё содержит цель EOS.

Создайте `models` для n=1,2,3 на одном `encoded["train"]`. Соберите DataFrame `comparison` с колонками `n`, `train_ppl`, `validation_ppl`, `train_targets`, `validation_targets`. Выберите `selected_n` по наименьшей validation perplexity, при равенстве — меньший n. Сохраните модель в `selected`, её тестовую оценку — в `test_result`.

Словарь и alpha остаются общими. Почему число целей в одной части должно совпадать у всех трёх порядков?

In [10]:
def evaluate_ppl(model, sentences, alpha=0.1):
    # Ваш код
    total_nll = 0.0
    targets = 0
    for sentence in sentences:
        for history, target in events(sentence, model["n"]):
            p = probability(model, history, target, alpha)
            total_nll += -math.log(p)
            targets += 1

    if targets == 0:
        raise ValueError("нет целевых событий")

    mean_nll = total_nll / targets
    return {"ppl": math.exp(mean_nll), "mean_nll": mean_nll, "targets": targets}

# Ваш код
models = {}
rows = []
for n in [1, 2, 3]:
    model = fit_model(encoded["train"], n, vocab)
    models[n] = model
    train_result = evaluate_ppl(model, encoded["train"])
    validation_result = evaluate_ppl(model, encoded["validation"])
    rows.append({
        "n": n,
        "train_ppl": train_result["ppl"],
        "validation_ppl": validation_result["ppl"],
        "train_targets": train_result["targets"],
        "validation_targets": validation_result["targets"],
    })
comparison = pd.DataFrame(rows)
display(comparison)

selected_n = None
best_ppl = None
for row in rows:
    if best_ppl is None or row["validation_ppl"] < best_ppl:
        best_ppl = row["validation_ppl"]
        selected_n = row["n"]

selected = models[selected_n]
test_result = evaluate_ppl(selected, encoded["test"])
print("Выбран n =", selected_n)
print("Test:", test_result)

,n,train_ppl,validation_ppl,train_targets,validation_targets
0,1,9.694419,6.638790,527,189
1,2,5.248427,4.979994,527,189
2,3,5.044076,6.038190,527,189


Выбран n = 2
Test: {'ppl': 4.940421045685083, 'mean_nll': 1.5974505594878547, 'targets': 190}


In [11]:
assert comparison["train_targets"].nunique() == comparison["validation_targets"].nunique() == 1
assert test_result["targets"] == sum(len(s)+1 for s in encoded["test"])
assert math.isfinite(test_result["ppl"]) and test_result["ppl"] >= 1
assert evaluate_ppl(selected, [[]])["targets"] == 1
assert np.isclose(math.exp(-np.log([0.5, 0.25, 0.5]).mean()), 16**(1/3))

**Ответ.** Число целей одинаково у всех n (527 на train, 189 на validation), потому что предсказывается тот же набор слов и EOS. Меняется только длина истории, поэтому perplexity трёх моделей сравнимы.

## 5. Продолжаем префикс

Закончите `generate(model, prefix="как", mode="sample", seed=42, max_new=40)`. Преобразуйте префикс тем же словарём. На каждом шаге получите распределение по последним n−1 токенам; для униграммы история пустая. В greedy выбирайте максимум, в sample используйте `rng.choice` с вероятностями.

При EOS завершайте генерацию, иначе продолжайте до `max_new`. Верните `text` (префикс и продолжение), `tokens` (только новые токены без EOS), `stopped_by` ("EOS" или "limit") и `steps` (число новых предсказаний, включая EOS, если он появился). Оставляйте UNK видимым.

Соберите `generations`: один результат greedy и sampling с seed 7, 42 и 99. Сравните повтор с одним seed и результаты разных seed. Чем условия этого опыта отличаются от расчёта perplexity?

In [12]:
def generate(model, prefix="как", mode="sample", seed=42, max_new=40):
    # Ваш код
    n = model["n"]
    model_vocab = list(model["vocab"])
    rng = np.random.default_rng(seed)

    prefix_tokens = encode(tokenize(prefix))
    sequence = [BOS] * (n - 1) + prefix_tokens
    new_tokens = []
    stopped_by = "limit"
    steps = 0

    for step in range(max_new):
        if n > 1:
            history = tuple(sequence[len(sequence) - (n - 1):])
        else:
            history = ()
        probs = distribution(model, history)

        if mode == "greedy":
            index = int(np.argmax(probs))
        elif mode == "sample":
            index = rng.choice(len(model_vocab), p=probs)
        else:
            raise ValueError("mode должен быть 'greedy' или 'sample'")

        token = model_vocab[index]
        steps += 1
        if token == EOS:
            stopped_by = "EOS"
            break
        sequence.append(token)
        new_tokens.append(token)

    return {
        "text": " ".join(prefix_tokens + new_tokens),
        "tokens": new_tokens,
        "stopped_by": stopped_by,
        "steps": steps,
    }


generations = []
for mode, seed in [("greedy", 42), ("sample", 7), ("sample", 42), ("sample", 99)]:
    result = generate(selected, mode=mode, seed=seed)
    generations.append({
        "mode": mode,
        "seed": seed,
        "stopped_by": result["stopped_by"],
        "steps": result["steps"],
        "text": result["text"],
    })
generations_table = pd.DataFrame(generations)
pd.set_option("display.max_colwidth", None)
display(generations_table)

print("Повтор seed 42 совпал:", generate(selected, seed=42) == generate(selected, seed=42))

,mode,seed,stopped_by,steps,text
0,greedy,42,limit,40,как <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>
1,sample,7,EOS,4,как подать социальной стипендии
2,sample,42,EOS,26,как получить другую сообщить о <UNK> стипендии можно ли <UNK> <UNK> <UNK> после из продлить другую выплаты не <UNK> к ли можно ли узнать социальной стипендии
3,sample,99,EOS,31,как нужно ли <UNK> справка о к <UNK> <UNK> <UNK> <UNK> после <UNK> <UNK> <UNK> <UNK> <UNK> после <UNK> в <UNK> <UNK> <UNK> <UNK> не <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>


Повтор seed 42 совпал: True


In [13]:
assert generate(selected, seed=42) == generate(selected, seed=42)
for result in [generate(selected, mode="greedy"), generate(selected, seed=7)]:
    assert 1 <= result["steps"] <= 40
    assert result["stopped_by"] in {"EOS", "limit"}
    assert all(t in vocab and t not in {BOS, EOS} for t in result["tokens"])
    if result["stopped_by"] == "limit": assert len(result["tokens"]) == 40

**Ответ.** С одним seed результат повторяется, с разными seed продолжения расходятся. Perplexity считается по эталонным префиксам, а при генерации модель продолжает свои собственные токены.

## Состояние после нулевого входа

Следующая ячейка воспроизводит скалярную RNN из лекции. Параметры заданы вручную. Сначала предскажите знак второго состояния, затем сравните цепочку входов (1;0) с (0;0).

In [14]:
# Скалярная RNN из лекции: параметры заданы вручную, обучение не выполняется.
h = 0.0
for x in [1.0, 0.0]:
    h = np.tanh(0.5*h + x)
    print("input", x, "state", h)

input 1.0 state 0.7615941559557649
input 0.0 state 0.36339948438905245


## Разбор результатов

Объясните выбор порядка по полученной таблице. Сопоставьте perplexity с долей UNK: какие различия исходных слов модель ещё способна учитывать? Затем сравните два продолжения, указав, как они завершились и какие переходы выглядят странно.

Для примера RNN объясните, почему при втором нулевом входе состояние осталось ненулевым. Сдайте выполненный ноутбук с этими пояснениями; перед сдачей перезапустите ядро и выполните все ячейки по порядку.

### Мой разбор

**Выбор порядка.** На validation perplexity: n=1 — 6,64, n=2 — 4,98, n=3 — 6,04. Выбрана биграмма. У триграммы ниже perplexity на train (5,04), но на validation её истории из двух слов часто не встречались. Тестовая perplexity биграммы 4,94 на 190 целях.

**UNK и perplexity.** UNK заменил 58% слов train и 71% test. Модель отличает только 45 частых слов, остальные слились в один токен, и предсказывать UNK легко. Низкая perplexity здесь означает не хорошее понимание текста, а грубый словарь.

**Продолжение 1, greedy.** Остановилось по лимиту: 40 раз подряд UNK. После «как» чаще всего шёл UNK (6 из 14), после UNK — снова UNK (139 из 264, EOS только 55), поэтому greedy зациклился.

**Продолжение 2, sample seed 7.** «как подать социальной стипендии», остановка по EOS на 4-м шаге. Переход «подать → социальной» в train не встречался и выбран только благодаря сглаживанию, отсюда грамматическая ошибка. «социальной → стипендии» и «стипендии → EOS» — частые переходы.

**RNN.** h₁ = tanh(0,5·0 + 1) ≈ 0,7616, h₂ = tanh(0,5·0,7616 + 0) ≈ 0,3634. Второй вход нулевой, но в формулу входит прошлое состояние, поэтому h₂ ≠ 0. При входах (0;0) оба состояния равны нулю.